In [2]:
# ============================================================
# HealthifyAI — Mistral-7B LoRA Finetuning (Google Colab)
# Run each cell block in order on Colab with A100 / T4 GPU
# ============================================================


# ── CELL 1: Install dependencies ────────────────────────────
# Run this first, then restart runtime when prompted

!pip install -q --upgrade pip
!pip uninstall -y transformers trl peft bitsandbytes accelerate datasets
!pip install -q transformers==4.44.2
!pip install -q trl==0.9.6
!pip install -q peft==0.12.0
!pip install -q bitsandbytes==0.43.3
!pip install -q accelerate==0.33.0
!pip install -q datasets==2.21.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 29.2 MB/s eta 0:00:00
Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: peft 0.18.1
Uninstalling peft-0.18.1:
  Successfully uninstalled peft-0.18.1
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0
Found existing installation: datasets 4.0.0
Uninstalling datasets-4.0.0:
  Successfully uninstalled datasets-4.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchtune 0.6.1 requires datasets, which is not installed.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tobler 0.14.0 requires numpy>=2.0, but you

In [3]:
!pip install -q transformers==4.44.2 trl==0.9.6 peft==0.12.0 accelerate==0.33.0 datasets==2.21.0
!pip install -q bitsandbytes==0.45.5  # has prebuilt CUDA 12.x wheels, no triton issue

In [2]:
# ── CELL 2: Imports ─────────────────────────────────────────
import torch
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)

from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")



CUDA available: True
GPU: Tesla T4


In [2]:
# ── CELL 3: Load & prepare dataset ──────────────────────────
# Upload your CSV to Colab first, then run this cell

df = pd.read_csv("final_dataset_healthify1.csv")

# Drop rows with any nulls in the columns we use
df = df[["ingredients_text", "serving_size", "review", "disease_conditions"]].dropna()
df = df.reset_index(drop=True)

print(f"Dataset size: {len(df)} rows")
print(df.head(2))


Dataset size: 2188 rows
                                    ingredients_text     serving_size  \
0  Organic cheddar cheese (organic cultured paste...  28 g (0.25 cup)   
1  Concentrated apple juice,ascorbic acid (vitami...  71 g (0.25 cup)   

                                              review  \
0  This serving of organic cheddar cheese is a go...   
1  This concentrated apple juice is high in sugar...   

                                  disease_conditions  
0                              obesity, hypertension  
1  Diabetes: High sugar content may exacerbate bl...  


In [3]:
# ── CELL 4: Format into Mistral instruction template ─────────
# Structure: [INST] ingredients + serving + disease [/INST] review
# Loss is ONLY computed on the review (output) tokens — not the input

def format_sample(row):
    instruction = (
        f"Analyze the following food product and provide a detailed health review.\n\n"
        f"Ingredients: {row['ingredients_text']}\n"
        f"Serving Size: {row['serving_size']}\n"
        f"Relevant Conditions: {row['disease_conditions']}"
    )
    response = row["review"].strip()
    # Mistral instruction format
    return {
        "text": f"<s>[INST] {instruction} [/INST] {response}</s>"
    }

dataset = Dataset.from_pandas(df)
dataset = dataset.map(format_sample)
dataset = dataset.remove_columns(
    [c for c in dataset.column_names if c != "text"]
)

# 90/10 train-eval split
split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split["train"]
eval_dataset  = split["test"]

print(f"Train: {len(train_dataset)} | Eval: {len(eval_dataset)}")
print("\nSample formatted text:")
print(train_dataset[0]["text"][:400], "...")


Map:   0%|          | 0/2188 [00:00<?, ? examples/s]

Train: 1969 | Eval: 219

Sample formatted text:
<s>[INST] Analyze the following food product and provide a detailed health review.

Ingredients: Glutinous rice flour peanut soy sauce (water soynean wheat salt) seaweed sugar, modified tapioca starch
Serving Size: 30 g (1.06 oz)
Relevant Conditions: diabetes, allergies, hypertension [/INST] This product contains a mix of carbohydrates and protein sources, but also some concerning ingredients. The ...


tried to quantise but failed

In [4]:

# ── CELL 5: Load Mistral with 4-bit QLoRA ───────────────────
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"

# 4-bit quantization — fits on T4 (16GB) and A100 (40GB)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",           # NormalFloat4 — best quality at 4-bit
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,       # nested quantization saves extra ~0.4 bits/param
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token   # Mistral has no native pad token
tokenizer.padding_side = "right"            # pad right for causal LM training

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map={"": 0},
    trust_remote_code=True,
)
# Remove bnb_config entirely, load in half precision

# loading model in float16(entire 14GB model)
# model = AutoModelForCausalLM.from_pretrained(
#     MODEL_NAME,
#     torch_dtype=torch.float16,
#     device_map={"": 0},
#     trust_remote_code=True,
# )

# Required before applying LoRA to a quantized model
model = prepare_model_for_kbit_training(model)

print("Model loaded ✓")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Model loaded ✓


In [5]:
# ── CELL 6: Apply LoRA adapters ──────────────────────────────
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,                    # rank — higher = more capacity, more VRAM
    lora_alpha=32,           # scaling = alpha/r = 2 (standard)
    lora_dropout=0.05,
    bias="none",
    # Target all attention projections for best coverage
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("done dana done")
# Expected output: ~0.25% of 7B params = ~20M trainable


trainable params: 13,631,488 || all params: 7,261,655,040 || trainable%: 0.1877
done dana done


In [6]:
# ── CELL 7: Loss masking — only train on review tokens ───────
# DataCollatorForCompletionOnlyLM masks everything before [/INST]
# so the model learns to generate reviews, not memorize ingredients

response_template = "[/INST]"
collator = DataCollatorForCompletionOnlyLM(
    response_template=response_template,
    tokenizer=tokenizer,
)



In [9]:
# ── CELL 8: Training arguments ───────────────────────────────
OUTPUT_DIR = "./healthifyai-mistral-lora"

# training_args = TrainingArguments(
#     output_dir=OUTPUT_DIR,
#     num_train_epochs=3,              # 3 epochs is a good start for 2K samples
#     per_device_train_batch_size=4,   # increase to 8 if on A100
#     per_device_eval_batch_size=4,
#     gradient_accumulation_steps=4,   # effective batch = 4 * 4 = 16
#     learning_rate=2e-4,              # standard for LoRA finetuning
#     lr_scheduler_type="cosine",
#     warmup_ratio=0.05,
#     fp16=True,
#     bf16=False,                       # bfloat16 is more stable than fp16 on A100
#     logging_steps=20,
#     eval_strategy="steps",
#     eval_steps=100,
#     save_strategy="steps",
#     save_steps=100,
#     save_total_limit=2,              # keep only best 2 checkpoints
#     load_best_model_at_end=True,
#     report_to="none",                # set to "wandb" if you want W&B tracking
#     optim="adamw_torch",       # paged optimizer — reduces GPU memory spikes
#     group_by_length=True,            # batch similar-length sequences → faster training
# )

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,              # DO NOT change this
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    fp16=True,
    bf16=False,
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=200,                  # not 10 — too frequent, slows training
    save_strategy="steps",
    save_steps=200,                  # not 10
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to="none",
    optim="adamw_torch",
    group_by_length=True,
)



In [10]:
# ── CELL 9: Train ────────────────────────────────────────────
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=collator,
    max_seq_length=768,              # covers 99% of your samples (max review ~894 chars)
    dataset_text_field="text",
)

print("Starting training...")
trainer.train()
print("Training complete ✓")



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': max_seq_length, dataset_text_field. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:280: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:318: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(


Map:   0%|          | 0/1969 [00:00<?, ? examples/s]

Map:   0%|          | 0/219 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:408: UserWarning: You passed a tokenizer with `padding_side` not equal to `right` to the SFTTrainer. This might lead to some unexpected behaviour due to overflow issues when training a model in half-precision. You might consider adding `tokenizer.padding_side = 'right'` to your code.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:488: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Starting training...


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss
200,0.340200,0.434807


We detected that you are passing `past_key_values` as a tuple and this is deprecated and will be removed in v4.43. Please use an appropriate `Cache` class (https://huggingface.co/docs/transformers/v4.41.3/en/internal/generation_utils#transformers.Cache)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss
200,0.340200,0.434807


Training complete ✓


In [11]:
# ── CELL 10: Save LoRA adapter weights ───────────────────────
# This saves ONLY the LoRA adapter weights (~100MB), NOT the full 14GB model
# You load these on top of the base Mistral model at inference time

ADAPTER_SAVE_PATH = "./healthifyai-lora-adapter"
model.save_pretrained(ADAPTER_SAVE_PATH)
tokenizer.save_pretrained(ADAPTER_SAVE_PATH)

print(f"LoRA adapter saved to: {ADAPTER_SAVE_PATH}")
print("Files saved:")
import os
for f in os.listdir(ADAPTER_SAVE_PATH):
    size = os.path.getsize(os.path.join(ADAPTER_SAVE_PATH, f)) / (1024*1024)
    print(f"  {f}  ({size:.1f} MB)")


LoRA adapter saved to: ./healthifyai-lora-adapter
Files saved:
  tokenizer.json  (1.9 MB)
  tokenizer.model  (0.6 MB)
  special_tokens_map.json  (0.0 MB)
  tokenizer_config.json  (0.1 MB)
  adapter_model.safetensors  (52.0 MB)
  adapter_config.json  (0.0 MB)
  README.md  (0.0 MB)


In [12]:
# ── CELL 13 (OPTIONAL): Push adapter to Hugging Face Hub ─────
from huggingface_hub import notebook_login
notebook_login()   # paste your HF token

In [13]:
model.push_to_hub("adarsh6/healthifyai-mistral-lora")
tokenizer.push_to_hub("adarsh6/healthifyai-mistral-lora")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   1%|1         |  557kB / 54.6MB            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...tral-lora/tokenizer.model: 100%|##########|  587kB /  587kB            

CommitInfo(commit_url='https://huggingface.co/adarsh6/healthifyai-mistral-lora/commit/fe753e790cbec154d95924fbddd34e951d417824', commit_message='Upload tokenizer', commit_description='', oid='fe753e790cbec154d95924fbddd34e951d417824', pr_url=None, repo_url=RepoUrl('https://huggingface.co/adarsh6/healthifyai-mistral-lora', endpoint='https://huggingface.co', repo_type='model', repo_id='adarsh6/healthifyai-mistral-lora'), pr_revision=None, pr_num=None)

In [3]:
# ── CELL 11 (OPTIONAL): Merge LoRA into base model & save full model ──
# Do this ONLY if you want a single standalone model file (needs ~28GB disk)
# Skip this if you're using Hugging Face inference endpoints (adapter is enough)


# from peft import PeftModel
# from transformers import AutoModelForCausalLM
# import torch

# base_model = AutoModelForCausalLM.from_pretrained(
#     MODEL_NAME,
#     torch_dtype=torch.bfloat16,
#     device_map="auto"
# )
# merged_model = PeftModel.from_pretrained(base_model, ADAPTER_SAVE_PATH)
# merged_model = merged_model.merge_and_unload()   # fuse LoRA weights into base weights

# MERGED_SAVE_PATH = "./healthifyai-mistral-merged"
# merged_model.save_pretrained(MERGED_SAVE_PATH)
# tokenizer.save_pretrained(MERGED_SAVE_PATH)
# print(f"Merged model saved to: {MERGED_SAVE_PATH}")

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

BASE_MODEL = "mistralai/Mistral-7B-Instruct-v0.3"
ADAPTER_PATH = "./healthifyai-lora-adapter"   # 👈 change if local path

SAVE_PATH = "./healthifyai-baselora-merged"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,   # ✅ FIXED
    device_map="auto",
    trust_remote_code=True
)

merged_model = PeftModel.from_pretrained(model, ADAPTER_PATH)

print("Merging LoRA with base model...")
merged_model = merged_model.merge_and_unload()

print("Saving merged model...")

merged_model.save_pretrained(
    SAVE_PATH,
    safe_serialization=True,        # 🔥 safetensors
    max_shard_size="2GB"            # 🔥 avoids crash
)

tokenizer.save_pretrained(SAVE_PATH)

print("✅ DONE! Model saved at:", SAVE_PATH)



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Merging LoRA with base model...


/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:336: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


Saving merged model...
✅ DONE! Model saved at: ./healthifyai-baselora-merged


In [5]:
# Push to HF
from huggingface_hub import notebook_login
notebook_login()

# merged_model.push_to_hub("adarsh6/healthifyai-merged")
# tokenizer = AutoTokenizer.from_pretrained("./healthifyai-lora-adapter")
# tokenizer.push_to_hub("adarsh6/healthifyai-merged")
# print("Done ✓")


In [6]:
MODEL_PATH = "./healthifyai-baselora-merged"
REPO_NAME = "adarsh6/healthifyai-base-lora-merged"

model = AutoModelForCausalLM.from_pretrained(MODEL_PATH)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

model.push_to_hub(REPO_NAME)
tokenizer.push_to_hub(REPO_NAME)

Unused kwargs: ['_load_in_4bit', '_load_in_8bit', 'quant_method']. These kwargs are not used in <class 'transformers.utils.quantization_config.BitsAndBytesConfig'>.
`low_cpu_mem_usage` was None, now set to True since model is quantized.


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...crx1tk9/model.safetensors:   1%|          | 23.5MB / 4.14GB            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...poq0qbstp/tokenizer.model: 100%|##########|  587kB /  587kB            

CommitInfo(commit_url='https://huggingface.co/adarsh6/healthifyai-base-lora-merged/commit/f4752ef270f0b7088c1b9b85363c772f340c9e2c', commit_message='Upload tokenizer', commit_description='', oid='f4752ef270f0b7088c1b9b85363c772f340c9e2c', pr_url=None, repo_url=RepoUrl('https://huggingface.co/adarsh6/healthifyai-base-lora-merged', endpoint='https://huggingface.co', repo_type='model', repo_id='adarsh6/healthifyai-base-lora-merged'), pr_revision=None, pr_num=None)

In [7]:
# ── CELL 12: Quick inference test ────────────────────────────
from peft import PeftModel

def generate_health_review(ingredients: str, serving_size: str, disease: str) -> str:
    instruction = (
        f"Analyze the following food product and provide a detailed health review.\n\n"
        f"Ingredients: {ingredients}\n"
        f"Serving Size: {serving_size}\n"
        f"Relevant Conditions: {disease}"
    )
    prompt = f"<s>[INST] {instruction} [/INST]"

    inputs = tokenizer(prompt, return_tensors="pt").to(merged_model.device)

    with torch.no_grad():
        outputs = merged_model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode only the generated tokens (skip input prompt)
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)


# Test with a sample from your dataset
test_review = generate_health_review(
    ingredients="Sugar, Palm Oil, High-Fructose Corn Syrup, Sodium Benzoate, Red 40",
    serving_size="30g",
    disease="obesity, hypertension"
)
print("Generated Review:")
print(test_review)


Generated Review:
This food product contains a combination of unhealthy ingredients that can pose significant health risks when consumed regularly. The high content of added sugars (Sugar, High-Fructose Corn Syrup) contributes to excessive calorie intake and may lead to weight gain, obesity, and related conditions such as type 2 diabetes. The palm oil used is often hydrogenated or partially hydrogenated, which increases its saturated fat content and may contribute to high cholesterol levels and heart disease. Sodium benzoate is a preservative that has been linked to potential kidney damage in high doses. Red 40 is an artificial colorant associated with hyperactivity in children and possible cancer risks. Consuming this product on a regular basis could exacerbate existing conditions like hypertension due to its high sodium content. It is recommended to limit or avoid consuming foods containing these ingredients for optimal health.
